this notebook computes the 2-5Hz theta power in the 1s window around each button press on every MEG sensor, log baseline corrected by the average power in each frequency band during the [-1s,0s] window before each trial.
the data is represented as a scalp map. 

inputs = subject ID

import toolboxes

In [ ]:
import h5py
import mne 
import numpy as np
import matplotlib.pyplot as plt
import os

define variables

In [ ]:
sub_ID = input('subject ID:')
theta_freq = np.array([2,5])
press_time_wind = np.array([-0.5,0.5])
baseline_time_wind = np.array([-1,0])
BP_chan = ['UPPT002']
MEG_chan = list()
chan_delay = 2.8
trial_time = 14.8
sfreq = 1000
ch_types = ['meg'] * 275

load subject files for button press times data (in beh data) and the MEG channel data. 

variables:

id = subject ID number in 2 digit format (e.g. 05, 10)   
spliced_id = subject ID in single digit format (e.g. 5, 8)  
CHAN = channel data filepath  
CD = channel data epochs variable  
BP1 = filepath for button timings and presses array  
BP2 = filepath for cue duration array  

outputs:

  
PT = press times array  
PRES = button press IDs for each trial array  
MEG = MEG channel data epochs object  
BP = UPPT002 channel data epochs object

In [ ]:
def filename_loader(id,BP_chan,MEG_chan):
    subject_folder = "//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data"
    button_press_folder = '/beh_data/s'+str(id)
    BP1 = subject_folder + button_press_folder + "/behDataMEG.mat"
    BP2 = subject_folder + button_press_folder + "/targetMEG.mat"
   
    if sub_ID.startswith('0'):
        spliced_id = id[1:]
        EMG_file ='/s0'+str(spliced_id)+'/Subject'+str(spliced_id)+'_spm_mne-epo'
    else:
        EMG_file ='/s'+str(id)+'/Subject'+str(id)+'_spm_mne-epo'
   
    CHAN = subject_folder + EMG_file + '.fif'

    with h5py.File(BP1, 'r') as f:
        timings = f['A/timing'][:]
        presses = f['A/press'][:]
    timings = timings.T

    PRES = presses.T

    with h5py.File(BP2, 'r') as f:
        cuedur = f['T/cueDur'][:]
    cuedur = cuedur.T

    PT = (timings + np.tile(cuedur,5)) / 1000

    CD = mne.read_epochs(CHAN, preload=True)

    for i in CD.info['ch_names']:
        if i.startswith('M'):
            MEG_chan.append(i)

    MEG = CD.get_data(picks=MEG_chan)
    BP = CD.get_data(picks=BP_chan)


    return PT,PRES,MEG,BP

press_times, presses, MEG_data, BP_data = filename_loader(sub_ID,BP_chan,MEG_chan)

Reading \\rdp.arc.ucl.ac.uk\ritd-ag-project-rd01pz-dbush99\Katja Button Press Data\s05\Subject5_spm_mne-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =       0.00 ...   14800.00 ms
        0 CTF compensation matrices available
Reading \\rdp.arc.ucl.ac.uk\ritd-ag-project-rd01pz-dbush99\Katja Button Press Data\s05\Subject5_spm_mne-epo.fif ...


check all loaded data

In [ ]:
print(f'press times array: {press_times}')
print('\n')
print(f'press times shape: {press_times.shape}')
print('\n')
print(f'press sequence array:{presses}')
print('\n')
print(f'presses array shape {presses.shape}')

print(MEG_data.info)
print(f'MEG Channel Names: {MEG_data.info["ch_names"]}')

MEG_data.plot(n_epochs=1,n_channels=4,picks=MEG_chan[0:3],block=True,scalings=800,show=True,)

In [ ]:
info = mne.create_info(ch_names=MEG_chan, ch_types=ch_types, sfreq=sfreq)

n_epochs,n_channels,n_times = MEG_data.shape
data = MEG_data.transpose([1,0,2]).reshape((n_channels,n_epochs*n_times))

raw = mne.io.RawArray(data,info)

In [ ]:
def press_time_flattener(pt_arr):

    tally_1 = np.zeros(shape=(1,5))
    tally_2 = 0
    pt_shift = np.zeros(shape=pt_arr.shape)
    tct = np.zeros(240) + chan_delay

    for trial_ind in range(0,240):
        pt_shift[trial_ind] = press_times[trial_ind,:] + tally_1
        tct[trial_ind] = tct[trial_ind] + tally_2
        tally_1 += trial_time
        tally_2 += trial_time

    pt_shift = pt_shift.flatten()

    valid = np.isfinite(pt_shift)
    print(f"Removed {(~valid).sum()} missing/non-finite values")
    pt_shift = pt_shift[valid]

    return pt_shift,tct

press_times_shifted, trial_cutoff_times = press_time_flattener(press_times)

In [ ]:
def epochs_maker(r,events,wind):
    samples = np.round(np.array(events)*sfreq).astype(int)

    press_events = np.column_stack([
    samples,
    np.zeros(len(samples),dtype=int),
    np.zeros(len(samples),dtype=int)
    ])

    epoch = mne.Epochs(
        r,
        press_events,
        tmin = wind[0],
        tmax = wind[1],
        baseline = None,
        preload = True
    )

    return epoch

press_epochs = epochs_maker(raw, press_times_shifted, press_time_wind)

trial_epochs = epochs_maker(raw, trial_cutoff_times, baseline_time_wind)
